# Лабораторна робота №3 — Лінійна регресія для акумулятора

**Студент:** Люлько Станіслав · **Група:** МІТ-31 · **random_state:** 17

## Тема й мета
Побудувати та оцінити модель лінійної регресії для оцінювання стану заряду літій-іонного акумулятора за напругою. Відпрацювати поділ даних, обробку ознак у Pipeline, порівняння з DummyRegressor і аналіз залишків.

## Дані та задача
Використано експериментально визначену **OCV–SOC curve**: 1001 вимір для літій-іонного акумулятора, де SOC — state of charge (0–1), а OCV — напруга розімкнутого кола у вольтах. Джерело: [Zenodo](https://doi.org/10.5281/zenodo.6985321), запис [10.5281/zenodo.6985321](https://zenodo.org/records/6985321).

Щоб робота мала мінімально потрібний обсяг і залишалася компактною, взято 501 рівномірно розподілене спостереження з повної кривої (крок 0.002 SOC). Ціль — `soc_percent` (заряд, %). Ознаки — OCV, OCV² та OCV³: це три поліноміальні представлення виміряної напруги, які дають лінійній моделі змогу наблизити нелінійну OCV–SOC залежність. Дані для аналізу збережено у `battery_soc.csv`; вихідна крива — `battery_soc_curve_source.csv`.

**Формулювання задачі:** оцінити відсоток заряду акумулятора за експериментально виміряною напругою розімкнутого кола та її поліноміальними перетвореннями. Ідентифікаторів та ознак, утворених із цілі, у таблиці немає.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEED = 17
OUT = Path.cwd() if (Path.cwd() / 'battery_soc.csv').exists() else Path.cwd() / 'Lab3'


## 1. Завантаження та первинний аналіз
Перевіряємо форму таблиці, типи, описову статистику, пропуски, дублікати й розподіл цілі.

In [ ]:
data = pd.read_csv(OUT / 'battery_soc.csv')
print('shape:', data.shape)
print('\nТипи даних:\n', data.dtypes)
data.info()
display(data.describe().T)
print('Пропуски за стовпцями:\n', data.isna().sum())
print('Дублікати:', data.duplicated().sum())
print('SOC, %:\n', data['soc_percent'].describe())


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(data['soc_percent'], bins=25, color='teal', alpha=.85)
plt.title('Розподіл стану заряду акумулятора')
plt.xlabel('Стан заряду, %'); plt.ylabel('Кількість вимірювань')
plt.tight_layout(); plt.savefig(OUT / 'target_distribution.png', dpi=160); plt.show()


## 2. Ціль, ознаки, train/test та preprocessing
`voltage_v` — безпосередньо виміряна ознака. `voltage_squared` і `voltage_cubed` — поліноміальні похідні цієї ознаки, а не похідні від цілі. Вони не використовують SOC і тому не створюють витоку. Розділяємо дані до навчання будь-якого перетворення. У цьому наборі всі три ознаки числові; категоріальний Pipeline залишено в ColumnTransformer як універсальну частину рекомендованого каркаса.

In [ ]:
target = 'soc_percent'
numeric_features = ['voltage_v', 'voltage_squared', 'voltage_cubed']
categorical_features = []
X = data[numeric_features + categorical_features]
y = data[target]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

numeric_pipe = Pipeline([('imputer', SimpleImputer(strategy='median'))])
categorical_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])
preprocessing = ColumnTransformer([
    ('num', numeric_pipe, numeric_features),
    ('cat', categorical_pipe, categorical_features)
])
model = Pipeline([('preprocessing', preprocessing), ('regressor', LinearRegression())])
baseline = Pipeline([
    ('preprocessing', preprocessing),
    ('regressor', DummyRegressor(strategy='mean'))
])
model.fit(X_train, y_train)
baseline.fit(X_train, y_train)
pred = model.predict(X_test)
pred_base = baseline.predict(X_test)


## 3. Порівняння метрик
MAE та RMSE подані у відсоткових пунктах SOC. Менші MAE/RMSE і більший R² означають кращу якість. Від’ємний R² означав би, що модель гірша за базовий прогноз середнім.

In [ ]:
def metric_row(name, actual, estimate):
    return {'model': name, 'MAE_SOC_percentage_points': mean_absolute_error(actual, estimate),
            'RMSE_SOC_percentage_points': mean_squared_error(actual, estimate) ** 0.5,
            'R2': r2_score(actual, estimate)}
metrics = pd.DataFrame([metric_row('DummyRegressor (mean)', y_test, pred_base),
                        metric_row('LinearRegression', y_test, pred)])
display(metrics.round(3))
metrics.to_csv(OUT / 'metrics.csv', index=False)
residuals = y_test.to_numpy() - pred
pd.DataFrame({'actual_soc_percent': y_test.to_numpy(), 'predicted_soc_percent': pred,
              'baseline_predicted_soc_percent': pred_base, 'residual_percentage_points': residuals,
              'absolute_error_percentage_points': np.abs(residuals)}).to_csv(OUT / 'test_predictions.csv', index=False)


## 4. Фактичні значення та залишки

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, pred, alpha=.65, s=18)
lims = [min(y_test.min(), pred.min()), max(y_test.max(), pred.max())]
ax.plot(lims, lims, 'r--', label='ідеальний прогноз')
ax.set(title='Фактичний та прогнозований стан заряду', xlabel='Фактичний SOC, %', ylabel='Прогнозований SOC, %')
ax.legend(); fig.tight_layout(); fig.savefig(OUT / 'actual_vs_predicted.png', dpi=160); plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(pred, residuals, alpha=.65, s=18)
ax.axhline(0, color='red', linestyle='--')
ax.set(title='Залишки відносно прогнозованого SOC', xlabel='Прогнозований SOC, %', ylabel='Залишок (факт − прогноз), відсоткові пункти')
fig.tight_layout(); fig.savefig(OUT / 'residuals_vs_predicted.png', dpi=160); plt.show()


## 5. Інтерпретація результату

In [ ]:
lin = metrics.set_index('model').loc['LinearRegression']
dum = metrics.set_index('model').loc['DummyRegressor (mean)']
print(f"Лінійна модель: MAE={lin['MAE_SOC_percentage_points']:.3f}, RMSE={lin['RMSE_SOC_percentage_points']:.3f}, R²={lin['R2']:.4f}")
print(f"Baseline: MAE={dum['MAE_SOC_percentage_points']:.3f}, RMSE={dum['RMSE_SOC_percentage_points']:.3f}, R²={dum['R2']:.4f}")
print('Найбільші абсолютні похибки (відсоткові пункти SOC):')
display(pd.read_csv(OUT / 'test_predictions.csv').nlargest(10, 'absolute_error_percentage_points'))


### Висновок
На тестових 101 спостереженнях лінійна регресія отримала MAE **1.258**, RMSE **1.566** відсоткового пункту SOC і R² **0.9970**. DummyRegressor має MAE **24.786**, RMSE **28.691** і R² приблизно **0**. Поліноміальна лінійна модель значно перевершила baseline на відкладеній частині цієї кривої. Найбільша абсолютна похибка — близько **4.77** відсоткового пункту, біля низького SOC. Там модель також може прогнозувати від’ємний SOC, що фізично неможливо. Залишки невеликі, але в окремих SOC-діапазонах помітне легке зміщення; крайні значення SOC потребують особливої уваги.

Це компактна навчальна задача за експериментальною OCV–SOC кривою, а не оцінка SOC у реальному часі під навантаженням: напруга розімкнутого кола вимірюється в контрольованих умовах. Поліноміальні ознаки пов’язані між собою, а випадковий split не є перевіркою на незалежній батареї. Для подальшої роботи варто оцінювати модель на іншій комірці, температурі та профілі навантаження; для прогнозів застосувати обмеження фізичного діапазону SOC.

Підсумкові числа метрик і таблиця найбільших помилок формуються вище на тестовій вибірці. Лінійна модель порівнюється з прогнозом середнім за MAE, RMSE та R². Графік залишків дає змогу перевірити зміщення й нелінійну структуру.

Це компактна навчальна задача за експериментальною OCV–SOC кривою, а не оцінка SOC у реальному часі під навантаженням: напруга розімкнутого кола вимірюється в контрольованих умовах. Поліноміальні ознаки пов’язані між собою, а випадковий split не є перевіркою на незалежній батареї. Для подальшої роботи варто оцінювати модель на іншій комірці, температурі та профілі навантаження.

## Запитання для самоперевірки
1. Регресія прогнозує числову величину; класифікація — клас.
2. Train навчає параметри, test оцінює на даних, не використаних під час навчання.
3. Інакше статистики перетворень залежать від тестових даних.
4. Витік — потрапляння в ознаки інформації про ціль; наприклад, обчислити імпутер на всьому наборі до split.
5. Pipeline навчає кроки послідовно без витоку; ColumnTransformer застосовує різні кроки до різних стовпців.
6. DummyRegressor задає орієнтир, який має перевершити корисна модель.
7. MAE — середня абсолютна помилка в одиницях цілі, тут відсоткових пунктах SOC.
8. RMSE квадратично карає великі відхилення.
9. Від’ємний R² означає гірший результат, ніж константний прогноз середнім.
10. Тренд/кривизна або зміна розкиду залишків свідчать про систематичні недоліки моделі.
